[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-IA_Ed2026/blob/main/meca-ia/C16/E16-1_Laminoir.ipynb)

# Cage de laminoir : lire les contraintes et les déformations sans jauges

## Le contexte

Une cage de laminoir serre une bande métallique entre deux cylindres, par un vérin hydraulique. Le vérin travaille sous une pression qui change à chaque nouvelle consigne d'écartement des cylindres, et son huile s'échauffe ou se refroidit en cours de laminage. Le bureau d'études voudrait connaître le tenseur des contraintes et des déformations à l'alésage du vérin, l'endroit où il fatigue, sans poser de jauges sur chaque machine en service : un banc d'essai a relevé 120 séquences de laminage de 4 minutes, au pas de 2 secondes, avec des jauges, et le modèle doit les retrouver à partir de la pression, de la température de l'huile, de la consigne d'écartement et de la nuance de la bande. Les consignes se succèdent pendant que la paroi du vérin continue de chauffer : le transitoire et le régime permanent se mêlent, et ce que mesurent les jauges à un instant dépend de ce qui s'est passé avant.

Les grandeurs à prédire sont douze nombres à chaque pas de temps, six contraintes et six déformations : c'est un problème de régression à plusieurs sorties, sur des séries temporelles.

Défi : prédire les douze sorties et obtenir un R² moyen supérieur à celui du modèle de départ sur la base de test. Ce modèle de départ est une régression linéaire sur décalages : pour chaque pas, elle lit les 64 pas précédents (128 secondes) des trois signaux ; il atteint un R² moyen de 0,77. À vous de faire mieux avec un réseau récurrent à mémoire, un LSTM, qui garde un état d'un pas au suivant. La règle de découpage est fixée plus bas : la base de test est faite de séquences entières. La base de test sert à mesurer, jamais à apprendre ni à régler.

## Les données

Fichier `MecaIA-Laminoir.xlsx`, dans le dossier `donnees` du dépôt : 14 400 lignes, 24 colonnes. Une ligne est un pas de 2 secondes d'une séquence de laminage de 4 minutes.

| Colonne | Signification | Unité |
|---|---|---|
| `sequence` | identifiant de la séquence, de `s01` à `s120`, sert à découper et à ranger les pas, n'est pas une entrée | catégorie |
| `temps_s` | temps depuis le début de la séquence, pas de 2 s, sert à ranger les pas | s |
| `nuance` | nuance de la bande : `doux`, `mi_dur` ou `dur` | catégorie |
| `pression_MPa` | pression mesurée dans le vérin de serrage | MPa |
| `temperature_huile_C` | température mesurée de l'huile du vérin | °C |
| `consigne_ecartement_mm` | consigne d'écartement des cylindres | mm |
| `sigma_xx_MPa` à `sigma_yz_MPa` | six composantes du tenseur des contraintes à l'alésage, repère de la rosette | MPa |
| `eps_xx_um_m` à `eps_yz_um_m` | six composantes du tenseur des déformations au même point | µm/m |
| `force_x_kN` à `moment_z_kNm` | torseur des efforts de réaction à l'embase, six composantes, cibles du défi suivant, écartées ici | kN et kN·m |

Origine : jeu simulé, cage de laminoir (cédage de la cage, jeu et plasticité du vérin, contrainte thermique, flexion et torsion). Quelques températures d'huile ne sont pas relevées. Il enseigne une méthode, et aucune conclusion de laminage n'en découle.

### Étape 1 : charger le fichier et regarder sa forme

On lit le classeur, puis on vérifie ses dimensions, ses cases vides et la longueur des séquences avant toute modélisation : la forme d'un fichier se lit sur le fichier entier. Le dossier des données est cherché depuis le dossier du notebook, puis depuis un clone du dépôt ; ouvert seul, par exemple dans Colab, le notebook télécharge une fois le jeu depuis le dépôt public.

In [ ]:
from pathlib import Path
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import set_config

set_config(display="text")
GRAINE = 42
FICHIER = "MecaIA-Laminoir.xlsx"
URL_JEU = "https://raw.githubusercontent.com/philippe-amuzuga/Meca-IA_Ed2026/main/donnees/MecaIA-Laminoir.xlsx"

# Le dossier `donnees` du dépôt, cherché depuis le dossier du notebook, puis depuis un clone.
candidats = [Path("..", "..", "donnees"), Path("Meca-IA_Ed2026", "donnees")]
dossier = next((d for d in candidats if (d / FICHIER).is_file()), None)
if dossier is None:
    # Notebook ouvert seul (Colab) : copie unique du jeu depuis le dépôt public.
    dossier = Path("donnees")
    dossier.mkdir(exist_ok=True)
    try:
        urllib.request.urlretrieve(URL_JEU, dossier / FICHIER)
    except OSError as erreur:
        raise FileNotFoundError(
            "Jeu introuvable : clonez le dépôt, ou vérifiez la connexion pour le télécharger.") from erreur
df = pd.read_excel(dossier / FICHIER)

cibles = [c for c in df.columns if c.startswith(("sigma_", "eps_"))]
print("dimensions :", df.shape)
vides = df.isna().sum()
print("cases vides :", vides[vides > 0].to_dict())
longueurs = df.groupby("sequence").size()
print("séquences :", len(longueurs), "de", longueurs.min(), "à", longueurs.max(), "pas")
print("sorties à prédire :", len(cibles))
df.head(3)

Lecture du résultat : 14 400 lignes, 24 colonnes. Le jeu réunit 120 séquences de 120 pas chacune, soit 4 minutes à 2 secondes par pas. La température de l'huile compte 276 cases vides, qu'il faudra remplir. Les douze sorties à prédire sont six contraintes et six déformations.

### Reproductibilité

Un réseau de neurones part de poids tirés au hasard et lit ses exemples dans un ordre tiré au hasard. La cellule suivante fixe toutes les graines, demande des opérations de calcul déterministes, impose un seul fil de calcul et un jeu d'instructions donné. Avec elle, deux exécutions du notebook sur le même ordinateur donnent les mêmes poids et le même score. Chaque cellule qui construit un réseau rappelle la graine dans sa première ligne : on peut la rejouer seule sans changer le résultat. Sur un autre ordinateur, ou sur un processeur qui mêle deux types de cœurs, les arrondis peuvent différer et le score changer à la troisième décimale.

In [ ]:
# Réseaux de neurones : mêmes résultats à chaque exécution.
# À placer après la cellule de lecture (qui définit GRAINE),
# avant tout usage de TensorFlow.
import os

# Lues à l'import de TensorFlow : journal réduit, oneDNN limité
# au produit matriciel, jeu d'instructions fixé.
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["ONEDNN_MAX_CPU_ISA"] = "AVX2"

import numpy as np
import tensorflow as tf
import keras

# Graines, opérations déterministes et un seul fil de calcul.
keras.utils.set_random_seed(GRAINE)
tf.config.experimental.enable_op_determinism()
tf.config.threading.set_intra_op_parallelism_threads(1)
tf.config.threading.set_inter_op_parallelism_threads(1)

print("TensorFlow", tf.__version__, "| Keras", keras.__version__,
      "| NumPy", np.__version__)

### Étape 2 : mettre le test de côté

Deux pas voisins d'une même séquence se ressemblent presque trait pour trait : en tirant les lignes au hasard, on mettrait des pas voisins des deux côtés, et le test se confondrait avec l'apprentissage. On tire donc des séquences entières : un quart des séquences, avec leurs 120 pas, est mis de côté avant tout apprentissage, tout réglage et toute comparaison avec la cible. La graine 42 fixe le tirage : chacun obtient la même base de test, et les scores se comparent. Dans l'apprentissage restant, un cinquième des séquences est réservé à la validation : il servira à comparer les modèles, jamais le test. Le torseur, cible du défi suivant, ne sert pas d'entrée.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

entrees = ["sequence", "nuance", "pression_MPa", "temperature_huile_C", "consigne_ecartement_mm"]
X, y = df[entrees], df[cibles]
i_train, i_test = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=GRAINE).split(X, y, groups=df["sequence"]))
X_train, X_test, y_train, y_test = X.iloc[i_train], X.iloc[i_test], y.iloc[i_train], y.iloc[i_test]
i_fit, i_val = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=GRAINE).split(X_train, y_train, groups=X_train["sequence"]))
X_fit, X_val, y_fit, y_val = X_train.iloc[i_fit], X_train.iloc[i_val], y_train.iloc[i_fit], y_train.iloc[i_val]
for nom, partie in [("ajustement", X_fit), ("validation", X_val), ("test", X_test)]:
    print(f"{nom} : {partie['sequence'].nunique()} séquences, {len(partie)} pas")

### Étape 3 : regarder une séquence de l'apprentissage

La figure place, pour trois séquences de l'apprentissage, une par nuance, la consigne d'écartement, la pression et une des contraintes, `sigma_xx_MPa`, en fonction du temps. Les paliers de consigne se succèdent ; la pression et la contrainte réagissent.

In [ ]:
nuances = ["doux", "mi_dur", "dur"]
exemples = [X_train.loc[X_train["nuance"] == n, "sequence"].iloc[0] for n in nuances]
fig, axes = plt.subplots(3, 1, figsize=(6.4, 5.2), sharex=True)
for sequence, nuance, style, gris in zip(exemples, nuances, ["-", "--", ":"], ["0.0", "0.35", "0.6"]):
    pas = df[df["sequence"] == sequence]
    for ax, colonne in zip(axes, ["consigne_ecartement_mm", "pression_MPa", "sigma_xx_MPa"]):
        ax.plot(pas["temps_s"], pas[colonne], linestyle=style, color=gris, label=f"{sequence} ({nuance})")
for ax, nom in zip(axes, ["consigne (mm)", "pression (MPa)", "contrainte xx (MPa)"]):
    ax.set_ylabel(nom)
    ax.grid(alpha=0.3)
axes[-1].set_xlabel("temps (s)")
axes[0].legend(frameon=False, ncol=3, fontsize=8)
plt.tight_layout()
plt.show()

Lecture du résultat : la consigne avance par paliers de 40 à 120 secondes. La pression réagit à chaque changement de consigne par un pic qui s'amortit en quelques dizaines de secondes, et son niveau dépend de la nuance. La contrainte ne se contente pas de suivre la pression : elle dérive entre deux changements, parce que l'huile et la paroi du vérin se réchauffent lentement. Une même pression ne donne donc pas toujours la même contrainte : tout dépend de ce qui s'est passé avant, d'où l'intérêt d'un modèle qui garde le passé en mémoire.

### Étape 4 : préparer les séquences

Chaque séquence devient un tableau de 120 pas et six variables : les trois signaux et la nuance en indicatrices. Une température non relevée reprend la dernière valeur lue dans la même séquence ; en tête, elle reçoit la médiane du signal sur l’ajustement seul. Les sorties sont rangées de la même façon, en douze colonnes par pas. Les entrées et les sorties du réseau seront centrées et réduites avec la moyenne et l'écart type des séquences d'apprentissage seules.

In [ ]:
# Votre code :

### Étape 5 : construire le modèle de départ

Le modèle de départ est une régression linéaire sur décalages. Pour chaque pas, on lui donne les valeurs des trois signaux aux 64 pas précédents (128 secondes de passé, le premier pas répété avant le début) et la nuance : 195 variables. Un pipeline les met à l'échelle puis ajuste une régression ridge, sur les séquences d'apprentissage seules. Ce modèle regarde donc le passé, mais d'une façon linéaire et avec une fenêtre fixe.

In [ ]:
# Votre code :

### Étape 6 : mesurer sur la base de test

Le modèle prédit des séquences qu'il n'a jamais vues. Le R² est calculé sur les douze sorties à la fois : c'est la moyenne des douze R², chaque sortie pesant autant. On lit aussi le R² des six contraintes et des six déformations séparément. Un témoin sans apprentissage, qui répond toujours la moyenne de chaque sortie sur l'apprentissage, donne le point de comparaison le plus bas.

In [ ]:
from sklearn.metrics import r2_score

score_depart = r2_score(y_test, y_pred)
score_temoin = r2_score(y_test, np.tile(y_train.mean().to_numpy(), (len(y_test), 1)))
print(f"R² moyen des douze sorties : {score_depart:.3f}")
print(f"R² des six contraintes : {r2_score(y_test.iloc[:, :6], y_pred[:, :6]):.3f}")
print(f"R² des six déformations : {r2_score(y_test.iloc[:, 6:], y_pred[:, 6:]):.3f}")
print(f"R² du témoin (toujours la moyenne de l'apprentissage) : {score_temoin:.3f}")

Lecture du résultat : le R² moyen est de 0,77. La régression linéaire sur décalages tient mieux les contraintes (0,89) que les déformations (0,65). Le témoin, qui répond toujours la moyenne de l'apprentissage, n'explique rien : son R² est voisin de zéro, et négatif parce que cette moyenne n'est pas celle du test. C'est ce chiffre, mesuré sur des séquences que le modèle n'a jamais vues, que l'on a le droit d'annoncer.

## Pour aller plus loin : un réseau à mémoire

Le modèle de départ de l'étape 5 reste le score de référence du défi. Les cellules suivantes montrent comment le dépasser sans regarder le test pour choisir. La première règle la longueur de la fenêtre du modèle linéaire sur la validation : combien de passé faut-il lui montrer ? La deuxième construit un réseau LSTM, qui lit la séquence pas à pas et garde un état, sa mémoire, d'un pas au suivant, et compare deux tailles de mémoire sur la même validation ; la norme du gradient y est écrêtée à 1, pour qu'un pas trop grand ne dérègle pas l'apprentissage, ce qui arrive aux réseaux récurrents. La troisième superpose, pour une séquence de validation, la mesure et les deux modèles. La quatrième lit les R² des contraintes et des déformations. La dernière mesure le réseau retenu une seule fois sur le test. Les valeurs sont lues dans les sorties imprimées.

In [ ]:
# Votre code :

In [ ]:
# Votre code :

In [ ]:
# Votre code :

temps = 2 * np.arange(PAS)
fig, ax = plt.subplots(figsize=(6.4, 3.6))
ax.plot(temps, T_val[k, :, colonne], color="0.0", label="mesure")
ax.plot(temps, sortie_ridge[k, :, colonne], color="0.6", linestyle="--", label="régression linéaire sur décalages")
ax.plot(temps, sortie_lstm[k, :, colonne], color="0.3", linestyle=":", linewidth=2, label=f"LSTM à {unites_retenues} unités")
ax.set(xlabel="temps (s)", ylabel="contrainte xx (MPa)")
ax.legend(frameon=False, fontsize=8)
ax.grid(alpha=0.3)
plt.show()

In [ ]:
# Votre code :

In [ ]:
# Votre code :

Lecture du résultat : sur la validation, la régression linéaire passe de 0,56 sans aucun passé (un pas, la valeur du moment) à 0,59 avec 8 pas, 0,65 avec 32 pas et 0,66 avec 64 pas : le passé compte, et l'essentiel du gain est acquis dès 32 pas, soit plus d'une minute. Le LSTM à 16 unités, retenu par la validation, atteint 0,85 ; l'autre taille, 0,84. Le réseau garde ce dont il a besoin dans son état, sans fenêtre fixée à l'avance. Par famille de sorties, la régression linéaire atteint 0,89 sur les contraintes et 0,43 sur les déformations, le LSTM 0,90 et 0,81. Mesuré une seule fois sur le test, le LSTM atteint un R² moyen de 0,91, soit 0,14 de plus que le modèle de départ.

## Le score de référence

Le score publié est le R² moyen du modèle de départ de l'étape 5 sur la base de test. L'empreinte identifie cette base de test : si la vôtre diffère, le découpage n'est pas le même et les scores ne se comparent pas.

In [ ]:
import hashlib

score = r2_score(y_test, y_pred)
positions = ",".join(str(i) for i in sorted(X_test.index))
print("EMPREINTE_TEST =", hashlib.sha256(positions.encode("ascii")).hexdigest())
print("SCORE_REFERENCE r2 = %.2f" % score)

## Synthèse

1. Ce que mesurent les jauges à un instant dépend de ce qui s'est passé avant : une pression identique ne donne pas toujours la même contrainte, et un modèle qui ne regarde que l'instant courant plafonne (0,56 en validation).
2. La régression linéaire sur décalages montre le passé au modèle par une fenêtre fixe : plus la fenêtre est longue, meilleur est le score (0,66 à 64 pas en validation), mais la loi reste linéaire (0,77 sur le test).
3. Un LSTM lit la séquence pas à pas et garde un état d'un pas au suivant ; sa mémoire se règle par le nombre d'unités, comparé sur la validation (16 unités retenues).
4. Les séquences se découpent entières : le test est fait de séquences que ni l'apprentissage ni la validation n'ont vues, et il ne sert qu'une fois, au réseau retenu (0,91 ici).
5. La graine est rappelée dans la cellule qui construit le réseau : on la rejoue seule sans changer le résultat.